<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Módulo 2</h2><br/>
<h1>Semana 6 · Martes — SimpleImputer y Guardar Modelos</h1>
<h3>Manejo de nulos dentro del pipeline + persistencia con joblib</h3>
<br/>
    <b>Instructor:</b> Jesús Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## 🎯 Objetivos de hoy

Al final vas a poder:

1. Usar **`SimpleImputer`** para manejar nulos dentro de un pipeline.
2. Conocer las **4 estrategias de imputación** y cuándo usar cada una.
3. Construir un pipeline **completo** con: imputación + escalado + encoding + modelo.
4. **Guardar y cargar modelos** con `joblib` (persistencia profesional).
5. Hacer **predicciones en producción** desde un modelo guardado.
6. Resolver **3 ejercicios** prácticos.

> Requisito previo: la clase de ayer (Pipelines y ColumnTransformer).

# 1. El problema de los nulos en producción

Hasta ahora usábamos `df.dropna()` o `df.fillna(0)` **antes** de entrenar. El problema:

🚨 **En producción, los datos NUEVOS también pueden tener nulos**. Si tu pipeline no sabe qué hacer con nulos, **se rompe**.

Ejemplo real: un cliente nuevo no completó su `edad` en el formulario. Tu modelo en producción **falla** porque no sabe imputar.

**Solución:** poner la imputación DENTRO del pipeline con **`SimpleImputer`**. Así, cualquier dato nuevo se imputa automáticamente con la estrategia que aprendió del train.

# 2. `SimpleImputer` — las 4 estrategias

`SimpleImputer` recibe un parámetro `strategy=` con 4 opciones:

| Estrategia | ¿Qué hace? | Cuándo usar |
|---|---|---|
| `'mean'` | Rellena con la **media** de la columna | Numérica con distribución simétrica |
| `'median'` | Rellena con la **mediana** | Numérica con outliers (más robusta) |
| `'most_frequent'` | Rellena con el **valor más común** (moda) | Funciona en numéricas Y categóricas |
| `'constant'` | Rellena con un valor fijo (`fill_value=`) | Cuando quieres un marcador explícito como 0 o `'Unknown'` |

### 🧠 Regla práctica

- 🔢 **Numéricas → `median`** (más robusta ante outliers)
- 🏷️ **Categóricas → `most_frequent` o `constant` con `'Unknown'`**

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# DataFrame con nulos
df = pd.DataFrame({
    'edad':    [25, 30, np.nan, 45, 28, np.nan, 50],
    'ingreso': [1000, 1500, 2000, np.nan, 1200, 1800, np.nan],
    'ciudad':  ['Santiago', 'Valparaíso', np.nan, 'Santiago', 'Concepción', 'Santiago', np.nan]
})
print('Antes:')
print(df)
print(f'\nNulos: {df.isnull().sum().sum()}')

Antes:
   edad  ingreso      ciudad
0  25.0   1000.0    Santiago
1  30.0   1500.0  Valparaíso
2   NaN   2000.0         NaN
3  45.0      NaN    Santiago
4  28.0   1200.0  Concepción
5   NaN   1800.0    Santiago
6  50.0      NaN         NaN

Nulos: 6


In [2]:
# Imputer para numéricas — mediana
imp_num = SimpleImputer(strategy='median')
df[['edad', 'ingreso']] = imp_num.fit_transform(df[['edad', 'ingreso']])

print('Después de imputar numéricas con mediana:')
print(df)
print(f'\nValores aprendidos por el imputer: {imp_num.statistics_}')

Después de imputar numéricas con mediana:
   edad  ingreso      ciudad
0  25.0   1000.0    Santiago
1  30.0   1500.0  Valparaíso
2  30.0   2000.0         NaN
3  45.0   1500.0    Santiago
4  28.0   1200.0  Concepción
5  30.0   1800.0    Santiago
6  50.0   1500.0         NaN

Valores aprendidos por el imputer: [  30. 1500.]


In [3]:
# Imputer para categórica — most_frequent (o constant con 'Unknown')
imp_cat = SimpleImputer(strategy='most_frequent')
df[['ciudad']] = imp_cat.fit_transform(df[['ciudad']])

print('Después de imputar categórica con most_frequent:')
print(df)
print(f'\nValor más frecuente aprendido: {imp_cat.statistics_}')
print(f'\nNulos restantes: {df.isnull().sum().sum()} ✅')

Después de imputar categórica con most_frequent:
   edad  ingreso      ciudad
0  25.0   1000.0    Santiago
1  30.0   1500.0  Valparaíso
2  30.0   2000.0    Santiago
3  45.0   1500.0    Santiago
4  28.0   1200.0  Concepción
5  30.0   1800.0    Santiago
6  50.0   1500.0    Santiago

Valor más frecuente aprendido: ['Santiago']

Nulos restantes: 0 ✅


# 3. Pipeline COMPLETO — imputer + scaler + encoder + modelo

Ahora vamos a juntar todo lo aprendido. Pipeline profesional:

```
Numéricas:    SimpleImputer(median) → StandardScaler
Categóricas:  SimpleImputer(most_frequent) → OneHotEncoder
                            ↓
                        Modelo final
```

Esto se construye con un **Pipeline anidado dentro de un ColumnTransformer dentro de otro Pipeline**. Suena complejo pero es elegante.

In [4]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn import set_config

set_config(display='diagram')

# Vamos a usar penguins — tiene nulos REALES en varias columnas
pen = sns.load_dataset('penguins')   # ojo: NO hago dropna()
print(f'Shape: {pen.shape}')
print(f'Nulos por columna:')
print(pen.isnull().sum())

Shape: (344, 7)
Nulos por columna:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


In [9]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69 entries, 194 to 16
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            69 non-null     object 
 1   island             69 non-null     object 
 2   bill_length_mm     68 non-null     float64
 3   bill_depth_mm      68 non-null     float64
 4   flipper_length_mm  68 non-null     float64
 5   sex                66 non-null     object 
dtypes: float64(3), object(3)
memory usage: 3.8+ KB


In [5]:
X = pen.drop(columns=['body_mass_g'])
y = pen['body_mass_g'].fillna(pen['body_mass_g'].median())  # también imputamos el target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipelines internos para cada tipo de columna
pipe_num = make_pipeline(SimpleImputer(strategy='median'),  StandardScaler())
pipe_cat = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))

preprocesador = make_column_transformer(
    (pipe_num, make_column_selector(dtype_include=np.number)),
    (pipe_cat, make_column_selector(dtype_include=object))
)

# Pipeline FINAL con el modelo
modelo = make_pipeline(preprocesador, LinearRegression())
modelo

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# Una sola línea — entrena imputación + scaling + encoding + modelo
modelo.fit(X_train, y_train)

print(f'\n💡 Nota: aunque X_train tiene NULOS, el pipeline los maneja AUTOMÁTICAMENTE.')
print(f'         Lo mismo va a pasar si llega un dato nuevo en producción con nulos.')

R² test: 0.8360

💡 Nota: aunque X_train tiene NULOS, el pipeline los maneja AUTOMÁTICAMENTE.
         Lo mismo va a pasar si llega un dato nuevo en producción con nulos.


In [10]:
print(f'R² test: {modelo.score(X_train, y_train):.4f}')

R² test: 0.8781


In [11]:
print(f'R² test: {modelo.score(X_test, y_test):.4f}')

R² test: 0.8360


# 4. Guardar y cargar modelos con `joblib`

## ¿Por qué necesitamos guardar el modelo?

Entrenar un modelo puede tardar **horas**. No quieres re-entrenarlo cada vez que tu app arranca. La solución es **persistir** el modelo entrenado en un archivo, y cargarlo cuando lo necesites.

### `joblib` vs `pickle`

| | `pickle` | `joblib` |
|---|---|---|
| Origen | Stdlib de Python | Recomendado por sklearn |
| Velocidad | Lenta con numpy | **Más rápida con numpy** |
| Uso | Genérico | Optimizado para sklearn |

👉 **Usa `joblib`** para modelos de sklearn — es el estándar profesional.

In [12]:
import joblib

# Guardar el pipeline completo (¡guarda TODO: imputers, scaler, encoder, modelo!)
joblib.dump(modelo, 'modelo_pinguinos.joblib')

print('✅ Modelo guardado en "modelo_pinguinos.joblib"')
print('   Este archivo se puede mover a producción, otro computador, otro servidor.')

✅ Modelo guardado en "modelo_pinguinos.joblib"
   Este archivo se puede mover a producción, otro computador, otro servidor.


In [14]:
# CARGAR el modelo desde disco (simulamos que estamos en otro script/servidor)
modelo_cargado = joblib.load('modelo_pinguinos.joblib')
modelo_cargado

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('pipeline-1', ...), ('pipeline-2', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:

# Verificamos que da el mismo resultado
print(f'R² del modelo cargado: {modelo_cargado.score(X_test, y_test):.4f}')
print('👉 Es exactamente el mismo que antes — está intacto.')

R² del modelo cargado: 0.8360
👉 Es exactamente el mismo que antes — está intacto.


In [19]:
# Simulamos un dato NUEVO llegando a producción (con nulos!)
pinguino_nuevo = pd.DataFrame([{
    'species':           'Adelie',
    'island':            'Torgersen',
    'bill_length_mm':    39.1,
    'bill_depth_mm':     np.nan,         # ← NULO en producción
    'flipper_length_mm': 181.0,
    'sex':               np.nan          # ← NULO en producción
}])

print('Pingüino nuevo (con nulos):')
print(pinguino_nuevo)


Pingüino nuevo (con nulos):
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  sex
0  Adelie  Torgersen            39.1            NaN              181.0  NaN


In [17]:
pinguino_nuevo

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,sex
0,Adelie,Torgersen,39.1,NaN,181.0,NaN


In [18]:
prediccion = modelo_cargado.predict(pinguino_nuevo)
print(f'\n🐧 Peso predicho: {prediccion[0]:.0f} gramos')
print('   ✅ El modelo manejó los nulos y predijo sin problema.')


🐧 Peso predicho: 3294 gramos
   ✅ El modelo manejó los nulos y predijo sin problema.


# 5. Buenas prácticas para producción

| Práctica | Por qué |
|---|---|
| **Guardar SIEMPRE el pipeline completo**, no solo el modelo | El modelo solo no sabe cómo escalar/codificar |
| **Versionar el archivo del modelo** (ej. `modelo_v1.joblib`) | Para volver atrás si algo falla |
| **Guardar también la versión de sklearn** usada | Cambios de versión pueden romper modelos |
| **Validar el output** antes de usarlo en producción | Detectar predicciones absurdas |
| **handle_unknown='ignore'** en OneHotEncoder | Para no romper si llega una categoría nueva |

---
# 🏋️ Ejercicios prácticos

## Ejercicio 1 — Pipeline con imputación sobre `mpg`

**Tarea:**

1. Cargar `sns.load_dataset('mpg')` — **NO** hagas dropna (tiene nulos reales).
2. Verificar que hay nulos en alguna columna numérica.
3. Target: `mpg`. Features: TODAS las demás (incluyendo `origin` categórica). Excluir `name`.
4. Construir un Pipeline con:
   - Para numéricas: `SimpleImputer(median)` + `StandardScaler`
   - Para categóricas: `SimpleImputer(most_frequent)` + `OneHotEncoder`
   - Modelo: `LinearRegression`
5. Train/test 80/20, entrenar y reportar R².

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer, make_column_selector

mpg = sns.load_dataset('mpg').drop(columns=['name'])
print('Nulos por columna:'); print(mpg.isnull().sum())

X = mpg.drop(columns=['mpg'])
y = mpg['mpg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe_num = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
pipe_cat = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))

preproc = make_column_transformer(
    (pipe_num, make_column_selector(dtype_include=np.number)),
    (pipe_cat, make_column_selector(dtype_include=object))
)

modelo = make_pipeline(preproc, LinearRegression()).fit(X_train, y_train)
print(f'\nR² test: {modelo.score(X_test, y_test):.4f}')
```
</details>

## Ejercicio 2 — Guardar y cargar un modelo

Vamos a simular el flujo completo de producción.

**Tarea:**

1. Usar el modelo entrenado del Ejercicio 1 (o entrenar uno nuevo).
2. **Guardarlo** en `modelo_mpg.joblib` con `joblib.dump()`.
3. **Cargarlo** en otra variable con `joblib.load()`.
4. Verificar que el R² del modelo cargado es **idéntico** al original.
5. Crear un **auto inventado** como DataFrame (con todas las features) — al menos una con nulo.
6. Hacer la predicción del consumo en mpg.

**Auto sugerido:**
```python
auto = pd.DataFrame([{
    'cylinders': 4, 'displacement': 150, 'horsepower': np.nan,
    'weight': 2800, 'acceleration': 16.0, 'model_year': 78, 'origin': 'usa'
}])
```

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución</summary>

```python
import pandas as pd
import numpy as np
import joblib

# Guardar
joblib.dump(modelo, 'modelo_mpg.joblib')
print('✅ Guardado')

# Cargar
modelo_cargado = joblib.load('modelo_mpg.joblib')
print(f'R² original: {modelo.score(X_test, y_test):.4f}')
print(f'R² cargado:  {modelo_cargado.score(X_test, y_test):.4f}  → ¡idéntico!')

# Predicción de auto nuevo (con nulo)
auto = pd.DataFrame([{
    'cylinders': 4, 'displacement': 150, 'horsepower': np.nan,
    'weight': 2800, 'acceleration': 16.0, 'model_year': 78, 'origin': 'usa'
}])
print(f'\nConsumo predicho: {modelo_cargado.predict(auto)[0]:.2f} mpg')
```
</details>

## Ejercicio 3 — DESAFÍO INTEGRADOR: Costos médicos 🏥💵

Eres data scientist en una compañía de seguros de salud. Te dan un dataset con información de pacientes y sus **costos médicos anuales**. Tu trabajo: construir un modelo predictivo profesional, listo para producción.

**Dataset:** `insurance.csv` (1,338 registros) — descárgalo así:

```python
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
df = pd.read_csv(url)
df.head()
```

### Las columnas

| Columna | Tipo | Descripción |
|---|---|---|
| `age` | numérica | Edad del cliente |
| `sex` | nominal | male / female |
| `bmi` | numérica | Índice de masa corporal |
| `children` | discreta ordinal | Número de hijos (0–5) |
| `smoker` | nominal binaria | yes / no |
| `region` | nominal | northeast / northwest / southeast / southwest |
| `charges` | numérica | **TARGET** — costo médico anual en USD |

### Tarea

#### Parte A — Exploración y limpieza (5 min)

1. Cargar el dataset.
2. Explorar: `.info()`, `.describe()`, `.isnull().sum()`, `.head()`.
3. **¿Tiene nulos? ¿Tiene duplicados?** Si tiene, decide qué hacer.

#### Parte B — Pipeline con 4 transformadores distintos

Construye un **ColumnTransformer** que aplique **CUATRO** transformaciones distintas:

| Transformador | Columnas | Por qué |
|---|---|---|
| `StandardScaler` | `age` | Distribución relativamente uniforme |
| `MinMaxScaler` | `bmi` | Para ver otro escalador en acción |
| `OrdinalEncoder` | `children` | Tiene orden natural (0 < 1 < 2 < ...) |
| `OneHotEncoder` | `sex`, `smoker`, `region` | Categóricas sin orden |

#### Parte C — Entrenar 3 modelos

Crea 3 pipelines con el MISMO ColumnTransformer pero distintos modelos:

1. `LinearRegression`
2. `RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)`
3. `GradientBoostingRegressor(n_estimators=200, random_state=42)`

#### Parte D — Guardar los 3 modelos

Con `joblib.dump()`, guarda los 3 modelos en archivos `.joblib`.

#### Parte E — Comparar los 3 modelos

1. Carga los 3 con `joblib.load()`.
2. Genera una **tabla comparativa** con R², MAE y RMSE de cada uno.
3. Haz un **barplot** del R².
4. **Decisión final:** ¿cuál pondrías en producción y por qué? (considera R², explicabilidad y tiempo).

In [ ]:
# Tu código aquí 👇



<details><summary>💡 Solución completa</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# === PARTE A: cargar y explorar ===
url = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
df = pd.read_csv(url)
print(f'Shape: {df.shape}')
print(f'Nulos: {df.isnull().sum().sum()}')
print(f'Duplicados: {df.duplicated().sum()}')
df = df.drop_duplicates().reset_index(drop=True)

# === PARTE B: ColumnTransformer con 4 transformadores ===
preprocesador = ColumnTransformer(transformers=[
    ('std',     StandardScaler(),  ['age']),
    ('minmax',  MinMaxScaler(),    ['bmi']),
    ('ord',     OrdinalEncoder(),  ['children']),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'), ['sex', 'smoker', 'region'])
])

X = df.drop(columns=['charges'])
y = df['charges']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === PARTE C: 3 pipelines con el MISMO preprocesador ===
modelos = {
    'lineal':           Pipeline([('prep', preprocesador), ('mod', LinearRegression())]),
    'random_forest':    Pipeline([('prep', preprocesador), ('mod', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))]),
    'gradient_boost':   Pipeline([('prep', preprocesador), ('mod', GradientBoostingRegressor(n_estimators=200, random_state=42))]),
}

# Entrenar y guardar cada uno
for nombre, pipe in modelos.items():
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, f'modelo_insurance_{nombre}.joblib')
    print(f'✅ Modelo {nombre} guardado')

# === PARTE D + E: cargar y comparar ===
resultados = []
for nombre in modelos:
    cargado = joblib.load(f'modelo_insurance_{nombre}.joblib')
    pred = cargado.predict(X_test)
    resultados.append({
        'Modelo': nombre,
        'R²':   r2_score(y_test, pred),
        'MAE':  mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred))
    })

tabla = pd.DataFrame(resultados).set_index('Modelo').round(2)
print('\nComparación de modelos:')
print(tabla)

# Barplot
fig, ax = plt.subplots(figsize=(9, 4))
colores = ['#4472C4', '#70AD47', '#ED7D31']
tabla['R²'].plot(kind='barh', color=colores, edgecolor='white', ax=ax)
for i, v in enumerate(tabla['R²']):
    ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')
ax.set_title('R² en test — Predicción de costos médicos', fontweight='bold')
ax.set_xlim(0, max(tabla['R²']) * 1.1)
plt.tight_layout(); plt.show()

# === Predicción de un cliente nuevo ===
cliente_nuevo = pd.DataFrame([{
    'age': 35, 'sex': 'female', 'bmi': 28.5,
    'children': 2, 'smoker': 'no', 'region': 'southeast'
}])

print('\n💵 Predicciones para cliente nuevo (35 años, mujer, no fumadora, 2 hijos):')
for nombre in modelos:
    m = joblib.load(f'modelo_insurance_{nombre}.joblib')
    print(f'  {nombre:18s} → ${m.predict(cliente_nuevo)[0]:,.0f} USD')

# 👉 Conclusión típica:
#    - GradientBoosting suele ganar (R² ≈ 0.88).
#    - RandomForest segundo (R² ≈ 0.85).
#    - Lineal tercero (R² ≈ 0.78).
#    
#    DECISIÓN DE NEGOCIO: si la interpretabilidad es clave (auditores, compliance),
#    Lineal con R²=0.78 puede ser suficiente. Si quieres máxima precisión para 
#    pricing dinámico, GradientBoosting es la opción.
```
</details>

---
## 📌 Cierre del día

Hoy aprendimos:

- ✅ **`SimpleImputer`** y sus 4 estrategias (`mean`, `median`, `most_frequent`, `constant`)
- ✅ Cómo incluir imputación **dentro** del pipeline → nulos manejados en producción
- ✅ Pipeline profesional completo: `Imputer + Scaler + Encoder + Modelo`
- ✅ **`joblib`** para guardar/cargar modelos
- ✅ Buenas prácticas de producción

### 🔜 Mañana — Miércoles 27

- **Random Forest y Bagging** — combinar muchos árboles para mejorar predicciones
- Feature Importance — qué variables pesan más para el modelo

Nos vemos 🚀